
<a target="_blank" href="https://colab.research.google.com/github/tuliosg/linguistica-computacional/blob/main/notebooks/1-Manipulando%20arquivos.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Manipulando arquivos**
Neste notebook, vamos entender como podemos acessar e manipular arquivos utilizando Python e bibliotecas específicas, como o pandas e pympi-ling.

### **Dependências**

In [ ]:
%pip install pympi-ling pandas openpyxl

In [13]:
import os

import pandas as pd
import pympi

In [ ]:
!mkdir -p ../content/data

!curl -L -o ../content/data/noticia_sintetica.txt https://raw.githubusercontent.com/tuliosg/linguistica-computacional/refs/heads/main/data/noticia_sintetica.txt
!curl -L -o ../content/data/ent_sintetica.eaf https://raw.githubusercontent.com/tuliosg/linguistica-computacional/refs/heads/main/data/ent_sintetica.eaf

## **Abrindo arquivos**
A ideia é a mesma de abrir um arquivo usando o explorador de arquivos, porém, aqui a gente precisa que o nosso computador consiga acessar o conteúdo do arquivo, não basta apenas nós sermos capazes de ler.

<img src="../docs/imgs/1-abrindo-arquivos.png" height="250">

### **Arquivos texto (*.txt*)**
Podemos abrir arquivos txt usando a função `open`, nativa do Python. Ao usarmos ela, precisamos passar o endereço do arquivo, o modo como queremos abrir e qual a codificação dele.  
* O endereço do arquivo trata da pasta onde ele está junto ao nome do arquivo. Nesse exemplo, podemos ver que o arquivo foi nomeado como `noticia_sintetica.txt` e está na pasta `data`
* Os modos de abertura podem ser:
  * `r` para leitura (de um arquivo existente)
  * `w` para escrita (de um arquivo existente)
  * `x` para criação e escrita de um arquivo novo
* A especificação da codificação é necessária para evitar erros no processamento de acentos e outros caracteres especiais. Para português brasileiro, utilizamos `utf-8`.

In [ ]:
# a variável conteudo_txt recebe tudo o que está no arquivo noticia_sintetica.txt
conteudo_txt = open('./data/noticia_sintetica.txt', 'r', encoding='utf-8').read()

# a função print serve apenas para mostrar o conteúdo armazenado na variável
print(conteudo_txt)

Pronto, a partir de agora, sempre que quisermos acessar o texto da notícia para fazermos qualquer operação, chamaremos a variável `conteudo_txt`.

### **Abrindo arquivos do ELAN (*.eaf*)**
Arquivos eaf são mais complexos do que texto simples, eles possuem trilhas, marcações de tempo e texto. Então, vamos precisar entender como acessar cada atributo desses.  
Para começar, abrimos o arquivo *eaf* utilizando a biblioteca `pympi-ling`, desenvolvida para lidar com arquivos do ELAN e do Praat. Após a abertura, a variável `conteudo_elan` passa a guardar tudo o que está presente no arquivo, tanto os conteúdos textuais quando as diversas marcações.  
A estrutura do arquivo eaf é baseada em xml, porém, quando lemos esse arquivo usando Python, convertemos a estrutura de xml para dicionário, no estilo `chave: valor`. Isso significa que cada trilha terá uma chave (a identificação da trilha) e seu valor, o conteúdo textual com a marcação de tempo.  
* As trilhas, contendo a identificação, os tempos e os conteúdos, podem ser acessadas usando o método `tiers`
* As marcações de tempo, em milissegundos, pode ser acessada através do método `timeslots`
* A identificação das trilhas pode ser acessada usando as `keys` do dicionário
  




In [ ]:
conteudo_elan = pympi.Elan.Eaf('./data/ent_sintetica.eaf')
        
tiers = conteudo_elan.tiers

print("É assim que fica o conteúdo do arquivo .eaf no formato de dicionário:\n")
display(tiers)

As marcações de tempo podem ser acessadas diretamente:

In [ ]:
timeslots = conteudo_elan.timeslots
display(timeslots)

✏️ **Observação:** por ser um dado sintético, as marcações de tempo está todas interligadas, contudo, num arquivo eaf real, haveriam diferenças entre o fim de uma marcação e o início de outra. 

A visualização dos identificadores das trilhas é útil para sabermos quantas trilhas há no arquivo e como foram nomeadas.

In [ ]:
tier_ids = list(tiers.keys())
display(tier_ids)

## **Processando o texto**
Para os nossos códigos, o texto é uma sequência de caraceteres, ou seja, uma lista de símbolos. E esse conceito é fundamental na hora do processamento.

In [ ]:
palavra = "teste"
print(list(palavra))

### **Segmentação**
Existem várias técnicas para a segmentação de textos, desde separações simples, tomando como base espaços em branco entre as palavras, até segmentações baseadas em aprendizado de máquina, que separam sentenças completas prevendo seu início e fim de acordo com o contexto. A escolha da melhor técnica sempre vai depender do objetivo.

### **O método `split`**
O `split` pega uma sequência de palavras e as separa de acordo com o argumento que você passar. Por exemplo: `split(".")` vai quebrar o texto partindo dos pontos.

In [ ]:
# o split sem nenhum argumento separa o texto em palavras, 
# considerando os espaços em branco como delimitadores
texto_separado = conteudo_txt.split()
display(texto_separado)

In [ ]:
# o split com o argumento '.' separa o texto em frases,
# considerando o ponto final como delimitador
frases = conteudo_txt.split('.')
display(frases)

Notem que, como o título da notícia não termina com ponto final, ele aparece junto da primeira frase do texto. Outro erro que esse tipo de delimitação poderia causar é a separação de termos como "Dra." ou "sr.", considerando o que vem antes do ponto como uma frase.

## **Convertendo tipos de arquivos**
Uma operação útil quando trabalhamos com dados é a conversão de tipo, por exemplo, transformar um arquivo .eaf em uma planilha. Esse tipo de manipulação pode permitir análises que seriam mais complexas no tipo inicial a depender do arquivo.

### **Convertendo o *eaf* em tabela**

In [19]:
def eaf_to_dataframe(arquivo):
    """
    Cria um DataFrame a partir de um dicionário de dados do ELAN e mapeia os 
    timestamps para seus valores numéricos.

    Args:
        arquivo (str): O caminho para o arquivo .eaf.

    Returns:
        pandas.DataFrame: Um DataFrame com as colunas 'entrevista', 'transcricao', 
        'tempo_inicio' e 'tempo_final'.
    """
    dados = []

    elan = pympi.Elan.Eaf(arquivo)
        
    tiers = elan.tiers
    timeslots = elan.timeslots
    tier_ids = list(tiers.keys())

    for tier_id, tier_data in tiers.items():
        anotacoes = tier_data[0]
        
        for chave_anotacao, valor_anotacao in anotacoes.items():
            id_tempo_inicio = valor_anotacao[0]
            id_tempo_final = valor_anotacao[1]
            
            tempo_inicio_mapeado = timeslots.get(id_tempo_inicio)
            tempo_final_mapeado = timeslots.get(id_tempo_final)

            linha = {
                'entrevista': os.path.basename(arquivo),
                'trilha': tier_id,
                'transcricao': valor_anotacao[2],
                'tempo_inicio': tempo_inicio_mapeado,
                'tempo_final': tempo_final_mapeado
            }
            dados.append(linha)

    df = pd.DataFrame(dados)

    colunas_ordenadas = ['entrevista', 'trilha', 'transcricao', 'tempo_inicio', 'tempo_final']
    df = df[colunas_ordenadas]

    return df

In [ ]:
arquivo_eaf = './data/ent_sintetica.eaf'
df_resultado = eaf_to_dataframe(arquivo_eaf)

display(df_resultado)

df_resultado.to_excel('./data/ent_sintetica.xlsx', index=False)

<div style="text-align: right; background-color: #f8f9fa; padding: 10px 15px; border-right: 4px solid #404040; border-radius: 5px; margin-top: 15px; display: inline-block; float: right; min-width: 300px;">
    <span style="font-weight: bold; color: #404040; font-size: 0.95em;">Métodos computacionais para linguística</span><br>
    <span style="color: #6c757d; font-size: 0.85em; font-style: italic;">Explorando fenômenos linguísticos</span><br>
    <span style="color: #595959; font-size: 0.8em; margin-top: 5px; display: inline-block;">Desenvolvido por <strong><a href="https://github.com/tuliosg" target="_blank" style="color: #404040; text-decoration: none;">Túlio Gois</a></strong></span>
</div>
<div style="clear: both;"></div>